# E-Commerce Profitability Analysis

### 1. Import Data and Explore

In [1]:
import pandas as pd
import numpy as np

#### 1.1 Load all three CSV

In [2]:
orders = pd.read_csv(r'D:\Data Analysis\1.Projects\6.E-Commerce Profitability Analysis\resources\orders.csv')

In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          2000 non-null   object 
 1   customer_id       2000 non-null   object 
 2   order_date        2000 non-null   object 
 3   channel           2000 non-null   object 
 4   payment_method    2000 non-null   object 
 5   region            2000 non-null   object 
 6   items_ordered     2000 non-null   int64  
 7   primary_category  2000 non-null   object 
 8   gross_revenue     2000 non-null   float64
 9   discount_pct      2000 non-null   int64  
 10  discount_amount   2000 non-null   float64
 11  shipping_cost     2000 non-null   float64
 12  product_cost      2000 non-null   float64
 13  platform_fee      2000 non-null   float64
 14  transaction_fee   2000 non-null   float64
 15  returned          2000 non-null   object 
 16  refund_amount     2000 non-null   float64


In [5]:
marketing_spend = pd.read_csv(r'D:\Data Analysis\1.Projects\6.E-Commerce Profitability Analysis\resources\marketing_spend.csv')

In [11]:
marketing_spend.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   month               144 non-null    object 
 1   platform            144 non-null    object 
 2   spend               144 non-null    float64
 3   impressions         144 non-null    int64  
 4   clicks              144 non-null    int64  
 5   conversions         144 non-null    int64  
 6   revenue_attributed  144 non-null    float64
 7   cpc                 144 non-null    float64
 8   cpa                 144 non-null    float64
 9   roas                144 non-null    float64
dtypes: float64(5), int64(3), object(2)
memory usage: 11.4+ KB


In [5]:
products = pd.read_csv(r'D:\Data Analysis\1.Projects\6.E-Commerce Profitability Analysis\resources\products.csv')

In [6]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   product_id              207 non-null    object 
 1   product_name            207 non-null    object 
 2   category                207 non-null    object 
 3   sub_category            207 non-null    object 
 4   unit_cost               207 non-null    float64
 5   selling_price           207 non-null    float64
 6   shipping_cost_per_unit  207 non-null    float64
 7   weight_lbs              207 non-null    float64
 8   supplier                207 non-null    object 
dtypes: float64(4), object(5)
memory usage: 14.7+ KB


In [7]:
pd.set_option('display.max.rows', 200)
pd.set_option('display.max.columns', 30)

#### 1.2 Verify the Correctness of Total Costs

In [9]:
orders['verify'] = round(orders['product_cost'] + orders['platform_fee'] + orders['transaction_fee'] + orders['shipping_cost'] - orders['total_costs'],2)
orders[orders['verify'] != 0][['order_id','verify']]

,order_id,verify


In [ ]:
# All the total costs are caculated correctly

### 2.Category Profitability
2.1 Group orders by product category.

2.2 Calculate total revenue, total costs, total profit, and profit margin for each.

2.3 Identify the top and bottom performers.

In [18]:
orders_by_category = orders.groupby('primary_category')['order_id'].count().reset_index(name = 'number_of_orders')
orders_by_category.sort_values('number_of_orders',ascending = False,inplace=True)

In [19]:
orders_by_category

,primary_category,number_of_orders
2,Clothing,293
6,Sports,292
3,Electronics,267
7,Toys,257
4,Food & Beverage,247
1,Books,239
0,Beauty,205
5,Home & Kitchen,200


In [9]:
category_aggregation = orders.groupby('primary_category').agg(total_revenue = ('net_revenue','sum'),
                                                                          total_costs = ('total_costs','sum')).reset_index()
category_aggregation['total_profit'] = category_aggregation['total_revenue'] - category_aggregation['total_costs']
category_aggregation['margin_profit'] = round(category_aggregation['total_profit'] / category_aggregation['total_revenue'] * 100,2)

In [10]:
category_aggregation.sort_values('margin_profit',ascending = False, inplace = True)
category_aggregation

,primary_category,total_revenue,total_costs,total_profit,margin_profit
3,Electronics,44885.84,30912.38,13973.46,31.13
7,Toys,33595.68,24809.43,8786.25,26.15
5,Home & Kitchen,26359.76,19671.19,6688.57,25.37
4,Food & Beverage,30662.69,23070.92,7591.77,24.76
6,Sports,32329.83,24733.20,7596.63,23.50
2,Clothing,31383.13,25110.65,6272.48,19.99
0,Beauty,18254.07,15079.19,3174.88,17.39
1,Books,18847.37,16597.58,2249.79,11.94


In [ ]:
# The highest profitable category is Electronics, and the lowest is Books 

### 3. Channel Analysis
3.1 Group by sales channel

3.2 Compare average order value, average profit, and return rate across channels.

3.3 Factor in platform fees for Marketplace and Social Commerce.

In [ ]:
# Average Order Value = Total Revenue / Number of Orders
# Return Rate = Number of Return

In [8]:
orders.head(5)

,order_id,customer_id,order_date,channel,payment_method,region,items_ordered,primary_category,gross_revenue,discount_pct,discount_amount,shipping_cost,product_cost,platform_fee,transaction_fee,returned,refund_amount,net_revenue,total_costs,profit,Unnamed: 20
0,ORD-000001,C-0617,2025/6/24,Mobile App,Gift Card,Northeast,2,Electronics,261.39,0,0.00,22.28,92.30,0.00,7.88,No,0.0,261.39,122.46,138.93,0
1,ORD-000002,C-0720,2025/8/23,Website,Credit Card,Midwest,1,Beauty,24.02,0,0.00,3.98,10.75,0.00,1.00,No,0.0,24.02,15.73,8.29,0
2,ORD-000003,C-0189,2025/7/20,Website,Credit Card,Southeast,2,Toys,52.60,0,0.00,14.60,31.41,0.00,1.83,No,0.0,52.60,47.84,4.76,0
3,ORD-000004,C-0058,2025/7/17,Mobile App,Debit Card,Southeast,2,Sports,30.63,5,1.53,30.98,16.27,0.00,1.14,No,0.0,29.10,48.39,-19.29,0
4,ORD-000005,C-0580,2025/10/11,Marketplace,Credit Card,Northeast,1,Sports,11.99,25,3.00,10.30,6.88,1.35,0.56,No,0.0,8.99,19.09,-10.10,0


In [40]:
set(orders['returned'])

{'No', 'Yes'}

In [47]:
channel_aggregation = orders.groupby('channel').agg(average_of_value = pd.NamedAgg(column = 'net_revenue',aggfunc='mean'),
                                                    average_profit = pd.NamedAgg(column ='profit',aggfunc='mean'),
                                                    number_orders = pd.NamedAgg(column ='order_id',aggfunc='count'),
                                                    number_returns = pd.NamedAgg(column = 'returned', aggfunc= lambda x: (x == "Yes").sum())).reset_index()
channel_aggregation['return_rate'] = channel_aggregation['number_returns'] / channel_aggregation['number_orders'] * 100
channel_aggregation = channel_aggregation.round(2)

In [48]:
channel_aggregation.sort_values('average_profit', ascending= False, inplace=True)

In [49]:
channel_aggregation

,channel,average_of_value,average_profit,number_orders,number_returns,return_rate
1,Mobile App,122.06,36.32,589,43,7.30
3,Website,116.97,31.60,795,56,7.04
2,Social Commerce,111.32,17.11,197,18,9.14
0,Marketplace,118.15,15.40,419,27,6.44
